# AMEX Enterprise Credit Risk Platform
## Notebook 14 — Executive Reports: Financial Impact & SMART Team Recommendations
### Phase 1 · Problem Statement 1: Credit Scoring / PD Prediction

CRISP-DM stage: **Deployment / Business Reporting**. Notebook 14 of 18. Depends on Notebook 01 (`project_config.json`) and Notebook 05 (`notebook_05_summary.json` / `model_comparison.csv` — the model's real, measured performance). Notebook 07's model-risk findings (`notebook_07_summary.json`) are used opportunistically if present, exactly like Notebook 06's output was opportunistic for Notebook 07 — not required.

**What this notebook does and does not compute.** This is a financial-impact and stakeholder-recommendation report, not a model-training notebook — it produces no new predictions. Two very different kinds of numbers appear in its output, and every table in every deliverable below labels which is which:

- **MEASURED (live, this run):** the champion model's real holdout AUC, AMEX competition metric, and Top-4% default-capture rate, read directly from Notebook 05's own saved results — plus the real, live portfolio default rate, recomputed by this notebook's own code directly from `train_labels.csv`, the same way Notebook 01 did (never carried over from memory).
- **ASSUMPTION (stated, editable):** the Kaggle AMEX dataset contains no revenue, cost, staffing, or investment figures for any real institution — it cannot. Every dollar figure in this report (portfolio size, exposure-at-default, staffing costs, cloud costs, revenue-per-account, etc.) rests on an explicit, clearly labeled hypothetical deployment scenario with stated assumption values, defined in Sections 4–5 below. These are illustrative industry-benchmark-style figures, not real AMEX financials — every constant is named, printed, and easy to edit to your own institution's real numbers. This report is a worked financial methodology, not investment or accounting advice.

**Deliverables:** `Financial_Impact_Report.docx` (methodology, assumptions table, projections, ROI/payback analysis, implementation plan, SMART recommendations for 10 stakeholder teams, 6 embedded charts) and `Financial_Impact_Dashboard.html` (a self-contained, interactive dashboard with a time-horizon slicer, a conservative/base/optimistic scenario selector, a searchable/filterable team-recommendations table, a sortable financial table, and 6 charts with real hover tooltips, data labels, legends, and titled axes — this HTML file loads the Chart.js library from a CDN, so an internet connection is needed the first time you open it in a browser).

**Honest note on "tooltips" across the two formats:** the interactive HTML dashboard has genuine hover tooltips (Chart.js). The static PNG charts embedded in the Word document cannot have hover tooltips — a printed image has no interactivity — so those instead carry printed data-value labels directly on the chart. Both satisfy the spirit of "self-explanatory chart" without overclaiming what a static image can do.

**Run the single code cell below, once.** Idempotent — every output file is written to a fixed path and overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOKS 01, 05 (07 OPTIONAL)
# =============================================================================
import os
import sys
import csv
import json
import math
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01, 05 (07 Optional)")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"  # preferred -- fallback below if absent
NB07_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_07_summary.json"  # optional -- used opportunistically only

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"{CONFIG_PATH} not found.\nFix: run 01_business_understanding.ipynb first.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DATA_ROOT = Path(PROJECT_CONFIG["data_root"])
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or PROJECT_CONFIG["hardware"].get("logical_cores_detected")
)

MODEL_DEV_DIR = PILLAR_DIRS["model_development"]
EXEC_DIR = PILLAR_DIRS["executive_reports"]
EXEC_DIR.mkdir(parents=True, exist_ok=True)
MODEL_COMPARISON_PATH = MODEL_DEV_DIR / "model_comparison.csv"  # Notebook 05's own fixed output path

# --- Champion identification: same resilient pattern used by Notebooks 06 and
#     07 -- prefer notebook_05_summary.json, fall back to reading
#     model_comparison.csv directly (same highest-holdout_amex_metric rule
#     Notebook 05 itself uses) when it isn't present. Either way, this notebook
#     only READS Notebook 05's saved numbers -- it never retrains or re-scores
#     anything. ---
NB05_SUMMARY = None
if NB05_SUMMARY_PATH.exists():
    with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
        NB05_SUMMARY = json.load(f)
    CHAMPION_NAME = NB05_SUMMARY["champion_model"]
    CHAMPION_METRICS = NB05_SUMMARY["champion_metrics"]
    _champion_source = NB05_SUMMARY_PATH.name
elif MODEL_COMPARISON_PATH.exists():
    with open(MODEL_COMPARISON_PATH, "r", encoding="utf-8", newline="") as _f:
        _cmp_rows = list(csv.DictReader(_f))
    if not _cmp_rows or "model" not in _cmp_rows[0] or "holdout_amex_metric" not in _cmp_rows[0]:
        raise RuntimeError(f"{MODEL_COMPARISON_PATH} exists but is missing the expected 'model' / "
                            f"'holdout_amex_metric' columns -- cannot identify a champion from it. "
                            f"Fix: re-run 05_model_development.ipynb.")
    _champion_row = max(_cmp_rows, key=lambda r: float(r["holdout_amex_metric"]))
    CHAMPION_NAME = _champion_row["model"]
    CHAMPION_METRICS = {k: float(v) if k != "model" else v for k, v in _champion_row.items()}
    _champion_source = f"{MODEL_COMPARISON_PATH.name} (fallback -- {NB05_SUMMARY_PATH.name} not found)"
else:
    raise FileNotFoundError(f"Neither {NB05_SUMMARY_PATH} nor {MODEL_COMPARISON_PATH} was found.\n"
                             f"Fix: run 05_model_development.ipynb first -- this notebook reports on its "
                             f"real, measured results; it has nothing to report without them.")

NB07_SUMMARY = None
if NB07_SUMMARY_PATH.exists():
    with open(NB07_SUMMARY_PATH, "r", encoding="utf-8") as f:
        NB07_SUMMARY = json.load(f)

print(f"Loaded config from     : {CONFIG_PATH}")
print(f"Champion model          : {CHAMPION_NAME}  (identified from: {_champion_source})")
print(f"Notebook 07 MRM output  : {'found -- will cross-reference risk tier / stability findings' if NB07_SUMMARY else 'not found -- those fields will show as Not Yet Completed (not required for this notebook)'}")
print(f"Executive report outputs will be written under: {EXEC_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONTEXT, LIBRARY IMPORTS & ADAPTIVE RAM CEILING
# =============================================================================
_section("SECTION 2: WARP Hardware Context, Library Imports & Adaptive RAM Ceiling")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    from docx import Document
    from docx.shared import Inches
except ImportError:
    missing.append("python-docx")

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


_process_start_rss_gb = _rss_gb()

# --- Adaptive RAM ceiling (same live-availability pattern as Notebooks 07-13):
#     checked here for reporting consistency across the platform, even though
#     this particular notebook's workload (reading two small CSV/JSON files,
#     writing a report) is I/O-bound, not memory-bound -- there is no
#     multi-gigabyte dataframe or model to size a ceiling around here. Stated
#     honestly below rather than pretending this notebook stresses RAM. ---
_live_vm = psutil.virtual_memory()
LIVE_TOTAL_RAM_BYTES = _live_vm.total
LIVE_AVAILABLE_RAM_BYTES = _live_vm.available
ADAPTIVE_RAM_FRACTION = _resource_limits.get("ram_fraction_cap", 0.90)
MAX_RAM_BYTES = int(LIVE_AVAILABLE_RAM_BYTES * ADAPTIVE_RAM_FRACTION)

print(f"WARP_THREAD_COUNT (read from config, reporting only -- this notebook does no Polars/model work): {WARP_THREAD_COUNT}")
print(f"Live available RAM right now : {LIVE_AVAILABLE_RAM_BYTES / 1e9:.1f} GB")
print(f"Adaptive RAM ceiling (90% of that): {MAX_RAM_BYTES / 1e9:.2f} GB -- not expected to be approached; "
      f"this notebook reads two small CSV/JSON files and one raw AMEX label file, and writes a report.")
print(f"GPU / NPU probe: not applicable -- this notebook performs no model inference or repeated array "
      f"computation of the kind Notebook 07's sensitivity analysis benchmarked; there is nothing here for a "
      f"GPU to meaningfully accelerate.")
print(f"\nProcess RSS at Section 2 start: {_process_start_rss_gb:.2f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: REAL MODEL-PERFORMANCE BASIS (MEASURED) + LIVE PORTFOLIO DEFAULT RATE
# =============================================================================
_section("SECTION 3: Real Model-Performance Basis (Measured) + Live Portfolio Default Rate")

# --- The only two numbers in this whole notebook that are genuinely MEASURED
#     rather than assumed: (1) the champion model's real holdout performance,
#     read directly from Notebook 05's own saved output -- not retyped by
#     hand; (2) the real portfolio default rate, recomputed HERE, live, by
#     this notebook's own code reading train_labels.csv directly -- the exact
#     same computation Notebook 01 performed, done independently so this
#     notebook never trusts a number carried over from memory or another
#     session, per this platform's zero-fabrication rule. ---
CHAMPION_HOLDOUT_AUC = float(CHAMPION_METRICS["holdout_auc"])
CHAMPION_AMEX_METRIC = float(CHAMPION_METRICS["holdout_amex_metric"])
CHAMPION_TOP4PCT_CAPTURE = float(CHAMPION_METRICS["holdout_top4pct_capture"])

labels_path = DATA_ROOT / "train_labels.csv"
if not labels_path.exists():
    raise FileNotFoundError(f"{labels_path} not found.\nFix: confirm DATA_ROOT (from project_config.json) "
                             f"still points at the folder containing the official AMEX CSVs.")

_target_sum, _row_count = 0, 0
with open(labels_path, "r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        _target_sum += int(row["target"])
        _row_count += 1

LIVE_PORTFOLIO_DEFAULT_RATE = _target_sum / _row_count
LIVE_TRAIN_CUSTOMER_COUNT = _row_count

print(f"MEASURED (Notebook 05, {_champion_source}):")
print(f"  Champion model                  : {CHAMPION_NAME}")
print(f"  Holdout AUC                     : {CHAMPION_HOLDOUT_AUC:.4f}")
print(f"  Holdout AMEX competition metric : {CHAMPION_AMEX_METRIC:.4f}")
print(f"  Holdout Top-4% default capture  : {CHAMPION_TOP4PCT_CAPTURE:.4%}  (share of real holdout defaulters "
      f"found within the riskiest 4% of scored customers)")
print(f"\nMEASURED (this notebook, live, from {labels_path.name}):")
print(f"  Real training population        : {LIVE_TRAIN_CUSTOMER_COUNT:,} customers")
print(f"  Real portfolio default rate      : {LIVE_PORTFOLIO_DEFAULT_RATE:.4%}")
if NB07_SUMMARY:
    print(f"\nMEASURED (Notebook 07, notebook_07_summary.json):")
    print(f"  Model risk tier                  : {NB07_SUMMARY.get('risk_tier', 'n/a')}")
    print(f"  PSI significant-shift features    : {NB07_SUMMARY.get('psi_significant_shift_features', 'n/a')}")
    print(f"  Rank-ordering inversions          : {NB07_SUMMARY.get('rank_ordering_inversions', 'n/a')}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: FINANCIAL SCENARIO ASSUMPTIONS (STATED -- HYPOTHETICAL DEPLOYMENT)
# =============================================================================
_section("SECTION 4: Financial Scenario Assumptions (Stated -- Hypothetical Deployment)")

# --- EVERY constant below is an explicit, editable ASSUMPTION -- illustrative,
#     industry-benchmark-style figures for a mid-size unsecured consumer-credit
#     card issuer, chosen to be defensible order-of-magnitude numbers, NOT real
#     American Express financials (the Kaggle dataset contains none). Edit
#     these to your own institution's real figures for a decision-grade
#     analysis -- everything downstream is computed live from whatever values
#     are set here. ---
SCENARIO_ASSUMPTIONS = {
    "deployment_portfolio_size_accounts": 2_000_000,
    "avg_exposure_at_default_usd": 3_500,
    "legacy_model_top4pct_capture_rate": 0.25,
    "collections_intervention_success_rate": 0.35,
    "new_applicants_per_year": 400_000,
    "incremental_approval_rate": 0.03,
    "avg_annual_net_interest_margin_per_account_usd": 220,
}
_assumption_notes = {
    "deployment_portfolio_size_accounts": "Illustrative mid-size card-issuer portfolio size.",
    "avg_exposure_at_default_usd": "Typical unsecured revolving-credit write-off exposure per defaulting account (industry benchmark range ~$2,500-$5,000).",
    "legacy_model_top4pct_capture_rate": "Assumed capture rate of a legacy rules-based/scorecard system being replaced -- conservative benchmark, deliberately lower than most modern gradient-boosted models to reflect a realistic 'before' state.",
    "collections_intervention_success_rate": "Share of flagged high-risk accounts where early intervention (credit-line reduction, proactive outreach, enhanced monitoring) actually prevents or reduces the eventual loss.",
    "new_applicants_per_year": "Assumed annual new-applicant volume for a portfolio this size.",
    "incremental_approval_rate": "Assumed percentage-point increase in approvals, at equivalent risk tolerance, among previously-marginal applicants -- enabled by better risk discrimination.",
    "avg_annual_net_interest_margin_per_account_usd": "Typical unsecured card annual net interest + fee margin per performing account (industry-benchmark range).",
}

print("ASSUMPTION (stated, editable -- not measured from the Kaggle dataset):")
for k, v in SCENARIO_ASSUMPTIONS.items():
    _label = k.replace("_", " ")
    _val = f"{v:,}" if isinstance(v, int) else (f"{v:.2%}" if v < 1 else f"{v:,.2f}")
    print(f"  {_label:<48}: {_val}")
    print(f"      -> {_assumption_notes[k]}")

_capture_delta_raw = CHAMPION_TOP4PCT_CAPTURE - SCENARIO_ASSUMPTIONS["legacy_model_top4pct_capture_rate"]
CAPTURE_RATE_IMPROVEMENT = max(0.0, _capture_delta_raw)
if _capture_delta_raw < 0:
    print(f"\n\u26a0\ufe0f  This run's champion measured Top-4% capture ({CHAMPION_TOP4PCT_CAPTURE:.2%}) is BELOW "
          f"the assumed legacy baseline ({SCENARIO_ASSUMPTIONS['legacy_model_top4pct_capture_rate']:.2%}) -- "
          f"clamping the capture-rate improvement used in the loss-avoided calculation to 0.0 rather than "
          f"reporting a negative benefit. This is expected on small/synthetic test data and should not occur "
          f"once this notebook is run against the real, fully-trained champion model.")
else:
    print(f"\nMeasured capture-rate improvement over the assumed legacy baseline: "
          f"{CAPTURE_RATE_IMPROVEMENT:+.2%} percentage points")

print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: INVESTMENT & COST ASSUMPTIONS (STATED) + IMPLEMENTATION TIMELINE
# =============================================================================
_section("SECTION 5: Investment & Cost Assumptions (Stated) + Implementation Timeline")

INVESTMENT_BREAKDOWN = [
    {"category": "Data Science / ML Engineering (4 FTE x 4 months)", "amount_usd": 224_000, "type": "one_time"},
    {"category": "Independent Model Risk Validation (1 FTE x 6 weeks)", "amount_usd": 24_000, "type": "one_time"},
    {"category": "Compliance & Legal Review (Basel/IFRS9, fair-lending sign-off)", "amount_usd": 40_000, "type": "one_time"},
    {"category": "MLOps / Platform Engineering (2 FTE x 3 months)", "amount_usd": 78_000, "type": "one_time"},
    {"category": "Change Management & Staff Retraining", "amount_usd": 30_000, "type": "one_time"},
    {"category": "Cloud Infrastructure Setup & Initial Compute", "amount_usd": 25_000, "type": "one_time"},
]
TOTAL_ONE_TIME_INVESTMENT_USD = sum(r["amount_usd"] for r in INVESTMENT_BREAKDOWN)

RECURRING_COSTS = [
    {"category": "Cloud Hosting / Inference Infrastructure", "amount_usd_per_month": 6_000},
    {"category": "Ongoing Monitoring & MRM Oversight (0.5 FTE)", "amount_usd_per_month": 8_000},
    {"category": "Quarterly Model Retraining (amortized monthly)", "amount_usd_per_month": 2_500},
]
TOTAL_MONTHLY_RECURRING_COST_USD = sum(r["amount_usd_per_month"] for r in RECURRING_COSTS)

IMPLEMENTATION_PHASES = [
    {"phase": "1. Discovery & Scoping", "chart_label": "1. Discovery & Scoping", "weeks": 2},
    {"phase": "2. Data Engineering Hardening (productionize Notebooks 02/04 pipelines)",
     "chart_label": "2. Data Engineering Hardening", "weeks": 3},
    {"phase": "3. Model Finalization & Sign-off (Notebook 05)",
     "chart_label": "3. Model Finalization & Sign-off", "weeks": 2},
    {"phase": "4. Model Risk Validation & Compliance Review (Notebooks 07/08)",
     "chart_label": "4. MRM & Compliance Review", "weeks": 4},
    {"phase": "5. MLOps Build-Out & Deployment (Notebooks 09-11)",
     "chart_label": "5. MLOps Build-Out & Deployment", "weeks": 5},
    {"phase": "6. UAT & Shadow-Mode Parallel Run", "chart_label": "6. UAT & Shadow-Mode", "weeks": 4},
    {"phase": "7. Go-Live & Hypercare", "chart_label": "7. Go-Live & Hypercare", "weeks": 2},
]
_start = 0
for _p in IMPLEMENTATION_PHASES:
    _p["start_week"] = _start
    _start += _p["weeks"]
IMPLEMENTATION_WEEKS = sum(p["weeks"] for p in IMPLEMENTATION_PHASES)
IMPLEMENTATION_MONTHS_CEIL = math.ceil(IMPLEMENTATION_WEEKS / 4.345)

investment_df = pd.DataFrame(INVESTMENT_BREAKDOWN)
investment_path = EXEC_DIR / "investment_breakdown.csv"
investment_df.to_csv(investment_path, index=False)

timeline_df = pd.DataFrame(IMPLEMENTATION_PHASES)
timeline_path = EXEC_DIR / "implementation_timeline.csv"
timeline_df.to_csv(timeline_path, index=False)

print("ASSUMPTION -- one-time investment (stated):")
for r in INVESTMENT_BREAKDOWN:
    print(f"  {r['category']:<62}: ${r['amount_usd']:>10,}")
print(f"  {'TOTAL ONE-TIME INVESTMENT':<62}: ${TOTAL_ONE_TIME_INVESTMENT_USD:>10,}")

print("\nASSUMPTION -- monthly recurring cost, post go-live (stated):")
for r in RECURRING_COSTS:
    print(f"  {r['category']:<62}: ${r['amount_usd_per_month']:>10,}/mo")
print(f"  {'TOTAL MONTHLY RECURRING COST':<62}: ${TOTAL_MONTHLY_RECURRING_COST_USD:>10,}/mo")

print(f"\nImplementation timeline (stated project plan, derived from the phases above):")
for r in IMPLEMENTATION_PHASES:
    print(f"  Week {r['start_week']:>2}-{r['start_week'] + r['weeks']:<3} : {r['phase']}")
print(f"  TOTAL: {IMPLEMENTATION_WEEKS} weeks (~{IMPLEMENTATION_WEEKS / 4.345:.1f} months)")
print(f"\u2705 Saved -> {investment_path}")
print(f"\u2705 Saved -> {timeline_path}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: MONTHLY FINANCIAL MODEL -- 5-YEAR CASH FLOW, ROI & PAYBACK (COMPUTED LIVE)
# =============================================================================
_section("SECTION 6: Monthly Financial Model -- 5-Year Cash Flow, ROI & Payback")

# --- Every number in this section is COMPUTED, this run, from the MEASURED
#     model performance in Section 3 and the stated ASSUMPTION constants in
#     Sections 4-5 -- nothing here is a typed-in final answer. Change any
#     assumption above and every value below (including payback month and the
#     dashboard/report built from it) recomputes accordingly. ---
RAMP_MONTHS = 6          # stated assumption: real-world adoption ramps linearly over 6 months post go-live
TOTAL_MONTHS_SIMULATED = 60  # 5 years

ANNUAL_LOSS_AVOIDED_USD = (
    SCENARIO_ASSUMPTIONS["deployment_portfolio_size_accounts"]
    * LIVE_PORTFOLIO_DEFAULT_RATE
    * SCENARIO_ASSUMPTIONS["avg_exposure_at_default_usd"]
    * CAPTURE_RATE_IMPROVEMENT
    * SCENARIO_ASSUMPTIONS["collections_intervention_success_rate"]
)
ANNUAL_REVENUE_UPLIFT_USD = (
    SCENARIO_ASSUMPTIONS["new_applicants_per_year"]
    * SCENARIO_ASSUMPTIONS["incremental_approval_rate"]
    * SCENARIO_ASSUMPTIONS["avg_annual_net_interest_margin_per_account_usd"]
)
ANNUAL_TOTAL_BENEFIT_USD = ANNUAL_LOSS_AVOIDED_USD + ANNUAL_REVENUE_UPLIFT_USD
MONTHLY_FULL_BENEFIT_USD = ANNUAL_TOTAL_BENEFIT_USD / 12.0


def _build_monthly_model(benefit_multiplier: float = 1.0) -> "pd.DataFrame":
    """Builds the month-by-month cost/benefit/cumulative-net/ROI table. The
    benefit_multiplier lets Sections 6-10 (and the dashboard's own client-side
    JS) express a conservative/base/optimistic scenario by scaling the BENEFIT
    side only -- costs are treated as fixed regardless of scenario, since they
    are staffing/infrastructure commitments made up front, not realizations
    that vary with how well the deployment performs."""
    rows = []
    cum_cost, cum_benefit = 0.0, 0.0
    for month in range(1, TOTAL_MONTHS_SIMULATED + 1):
        if month <= IMPLEMENTATION_MONTHS_CEIL:
            month_cost = TOTAL_ONE_TIME_INVESTMENT_USD / IMPLEMENTATION_MONTHS_CEIL
            month_benefit = 0.0
        else:
            months_since_golive = month - IMPLEMENTATION_MONTHS_CEIL
            ramp_fraction = min(1.0, months_since_golive / RAMP_MONTHS)
            month_cost = TOTAL_MONTHLY_RECURRING_COST_USD
            month_benefit = MONTHLY_FULL_BENEFIT_USD * ramp_fraction * benefit_multiplier
        cum_cost += month_cost
        cum_benefit += month_benefit
        cum_net = cum_benefit - cum_cost
        roi_pct = (cum_net / cum_cost * 100.0) if cum_cost > 0 else None
        rows.append({
            "month": month, "month_cost_usd": round(month_cost, 2), "month_benefit_usd": round(month_benefit, 2),
            "cumulative_cost_usd": round(cum_cost, 2), "cumulative_benefit_usd": round(cum_benefit, 2),
            "cumulative_net_usd": round(cum_net, 2), "roi_pct": round(roi_pct, 2) if roi_pct is not None else None,
        })
    return pd.DataFrame(rows)


SCENARIO_MULTIPLIERS = {"conservative": 0.7, "base": 1.0, "optimistic": 1.3}
monthly_model_by_scenario = {name: _build_monthly_model(mult) for name, mult in SCENARIO_MULTIPLIERS.items()}
monthly_model_df = monthly_model_by_scenario["base"]  # base scenario is the notebook's headline table

monthly_model_path = EXEC_DIR / "monthly_financial_model.csv"
monthly_model_df.to_csv(monthly_model_path, index=False)


def _payback_month(df: "pd.DataFrame"):
    _positive = df[df["cumulative_net_usd"] > 0]
    return int(_positive.iloc[0]["month"]) if len(_positive) > 0 else None


PAYBACK_MONTH_BY_SCENARIO = {name: _payback_month(df) for name, df in monthly_model_by_scenario.items()}

HORIZONS = [("1 Month", 1), ("3 Months", 3), ("6 Months", 6), ("1 Year", 12),
            ("2 Years", 24), ("3 Years", 36), ("5 Years", 60)]

horizon_rows = []
for scenario_name, df in monthly_model_by_scenario.items():
    for label, m in HORIZONS:
        r = df[df["month"] == m].iloc[0]
        horizon_rows.append({
            "scenario": scenario_name, "horizon_label": label, "months": m,
            "cumulative_cost_usd": r["cumulative_cost_usd"], "cumulative_benefit_usd": r["cumulative_benefit_usd"],
            "cumulative_net_usd": r["cumulative_net_usd"], "roi_pct": r["roi_pct"],
        })
horizon_df = pd.DataFrame(horizon_rows)
horizon_path = EXEC_DIR / "financial_horizon_summary.csv"
horizon_df.to_csv(horizon_path, index=False)

print(f"ANNUAL_LOSS_AVOIDED_USD (computed)   : ${ANNUAL_LOSS_AVOIDED_USD:,.0f}")
print(f"ANNUAL_REVENUE_UPLIFT_USD (computed) : ${ANNUAL_REVENUE_UPLIFT_USD:,.0f}")
print(f"ANNUAL_TOTAL_BENEFIT_USD (computed)  : ${ANNUAL_TOTAL_BENEFIT_USD:,.0f}  (steady state, full ramp, base scenario)")
print(f"Implementation months (cost-only)    : {IMPLEMENTATION_MONTHS_CEIL}")
print(f"\nBase-scenario snapshot by horizon:")
print(horizon_df[horizon_df["scenario"] == "base"].drop(columns=["scenario"]).to_string(index=False))
print(f"\nPayback period by scenario (first month cumulative net benefit turns positive):")
for name, pm in PAYBACK_MONTH_BY_SCENARIO.items():
    print(f"  {name:<12}: {(str(pm) + ' months') if pm else 'not reached within 60 months'}")
print(f"\u2705 Saved -> {monthly_model_path}")
print(f"\u2705 Saved -> {horizon_path}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: SMART RECOMMENDATIONS -- 10 STAKEHOLDER TEAMS (GROUNDED IN REAL FINDINGS)
# =============================================================================
_section("SECTION 7: SMART Recommendations -- 10 Stakeholder Teams")

# --- The rubric/wording of each recommendation is authored, stated guidance --
#     same as Notebook 07's documented risk-tiering rubric -- but every NUMBER
#     quoted inside each recommendation is pulled from this run's own real
#     computed values (Sections 3, 6) or, where Notebook 07 hasn't been run
#     yet, honestly marked as pending rather than invented. ---
_risk_tier_text = NB07_SUMMARY.get("risk_tier") if NB07_SUMMARY else None
_psi_sig_text = NB07_SUMMARY.get("psi_significant_shift_features") if NB07_SUMMARY else None
_mrm_status = f"Notebook 07 assigned risk tier '{_risk_tier_text}'" if _risk_tier_text else \
    "Notebook 07 (Model Risk Management) has not been run yet in this session"
_base_5yr = horizon_df[(horizon_df["scenario"] == "base") & (horizon_df["horizon_label"] == "5 Years")].iloc[0]
_base_payback = PAYBACK_MONTH_BY_SCENARIO["base"]

SMART_RECOMMENDATIONS = [
    {"team": "Executive Leadership / CRO", "specific": "Approve production deployment funding based on the modeled 5-year net benefit.",
     "measurable": f"5-year cumulative net benefit ${_base_5yr['cumulative_net_usd']:,.0f}, ROI {_base_5yr['roi_pct']:.0f}%, payback in {_base_payback or '60+'} months (base scenario).",
     "achievable": f"Champion model measured holdout AMEX metric {CHAMPION_AMEX_METRIC:.4f}, a real, validated result from Notebook 05.",
     "relevant": "Portfolio-wide default exposure and capital adequacy are the CRO's direct mandate.",
     "time_bound": "Funding decision within 1 sprint of this report; re-review assumptions annually."},
    {"team": "Head of Credit Risk / Underwriting", "specific": "Replace the legacy scorecard's top-4% flagging logic with the new champion model's output.",
     "measurable": f"Measured Top-4% default capture improves from an assumed {SCENARIO_ASSUMPTIONS['legacy_model_top4pct_capture_rate']:.0%} (legacy) to {CHAMPION_TOP4PCT_CAPTURE:.1%} (new model, measured).",
     "achievable": "No new data collection required -- Notebook 05's model scores directly on the existing feature store.",
     "relevant": "Directly drives the loss-avoided component of the financial case.",
     "time_bound": f"Cutover during Phase 5-6 of the implementation plan (weeks {IMPLEMENTATION_PHASES[4]['start_week']}-{IMPLEMENTATION_PHASES[5]['start_week'] + IMPLEMENTATION_PHASES[5]['weeks']})."},
    {"team": "Model Risk Management / Independent Validation", "specific": "Complete (or re-confirm) SR 11-7-style independent validation before go-live.",
     "measurable": f"{_mrm_status}.",
     "achievable": "Notebook 07 runs PSI, rank-ordering, sensitivity, and challenger-benchmark checks directly against Notebook 05's saved champion.",
     "relevant": "Regulatory and internal governance prerequisite for any credit-decisioning model.",
     "time_bound": "Must complete during Phase 4 (Model Risk Validation & Compliance Review) before Phase 5 begins."},
    {"team": "Compliance / Regulatory Affairs", "specific": "Complete Basel III / IFRS 9 capital-treatment mapping and fair-lending review.",
     "measurable": f"{'0' if _psi_sig_text == 0 else (_psi_sig_text if _psi_sig_text is not None else 'pending')} feature(s) flagged with significant population shift" + (" (Notebook 07)." if NB07_SUMMARY else " -- pending Notebook 07."),
     "achievable": "Notebook 08 (Basel III / IFRS 9 Mapping) produces the required documentation trail.",
     "relevant": "Deployment cannot proceed at a regulated card issuer without this sign-off.",
     "time_bound": "Complete within Phase 4, in parallel with Model Risk Management."},
    {"team": "Collections Operations", "specific": "Build the intervention workflow (outreach, credit-line adjustment) for accounts newly flagged by the improved model.",
     "measurable": f"Assumed {SCENARIO_ASSUMPTIONS['collections_intervention_success_rate']:.0%} intervention success rate drives ${ANNUAL_LOSS_AVOIDED_USD:,.0f}/year in modeled loss avoidance -- track actual realized success rate against this assumption post go-live.",
     "achievable": "Workflow changes only, no new systems required beyond the model's output feed.",
     "relevant": "Realizing the loss-avoided benefit depends entirely on Collections actually acting on the model's flags.",
     "time_bound": "Operational by end of Phase 6 (UAT & Shadow-Mode)."},
    {"team": "Data Engineering", "specific": "Productionize the Notebook 02/04 Polars feature-store pipeline for daily/real-time scoring.",
     "measurable": "Pipeline must reproduce the same feature set the champion model was trained on (see Notebook 04's engineered feature columns).",
     "achievable": "Pipeline already exists and runs end-to-end in this platform; this is a hardening/scheduling task, not new development.",
     "relevant": "Every downstream number in this report assumes the production feature set matches what Notebook 05 trained on.",
     "time_bound": "Complete during Phase 2 (weeks 2-5)."},
    {"team": "MLOps / Platform Engineering", "specific": "Stand up the FastAPI scoring service, Docker packaging, and monitoring pipeline (Notebooks 09-12).",
     "measurable": f"Monthly recurring cost budget: ${TOTAL_MONTHLY_RECURRING_COST_USD:,}/month post go-live (assumption, Section 5).",
     "achievable": "Architecture and monitoring triggers (PSI > 0.25) are already defined in this platform's governance docs.",
     "relevant": "Deployment reliability and drift monitoring directly protect the modeled financial benefit from decaying silently.",
     "time_bound": "Complete during Phase 5 (weeks 9-14)."},
    {"team": "Card Issuing / Product Business Unit", "specific": "Adjust approval-policy thresholds to capture the modeled incremental-approval opportunity.",
     "measurable": f"${ANNUAL_REVENUE_UPLIFT_USD:,.0f}/year modeled revenue uplift from a stated {SCENARIO_ASSUMPTIONS['incremental_approval_rate']:.0%}-point increase in approvals among {SCENARIO_ASSUMPTIONS['new_applicants_per_year']:,} assumed annual applicants.",
     "achievable": "Requires only a policy-threshold change once the new model's calibration is validated (Notebook 07).",
     "relevant": "New-business revenue is the second major component of the financial case, alongside loss avoidance.",
     "time_bound": "Threshold change effective at go-live (Phase 7)."},
    {"team": "Finance / CFO Office", "specific": "Track realized costs and benefits against this report's modeled figures on a monthly cadence.",
     "measurable": f"Total one-time investment ${TOTAL_ONE_TIME_INVESTMENT_USD:,}; monthly recurring ${TOTAL_MONTHLY_RECURRING_COST_USD:,}; base-scenario payback at month {_base_payback or '60+'}.",
     "achievable": "monthly_financial_model.csv (this notebook's own output) gives a ready-made tracking template.",
     "relevant": "Validates or corrects this report's assumptions with real realized figures as they become available.",
     "time_bound": "First realized-vs-modeled variance review at month 3 post go-live, then quarterly."},
    {"team": "External Regulator Engagement (e.g. Federal Reserve / OCC)", "specific": "Maintain the full documentation trail (Notebooks 01, 05, 07, 08, 14) as supervisory evidence.",
     "measurable": "9 governance dimensions tracked in Notebook 07's checklist" + (f" -- current tier: {_risk_tier_text}." if _risk_tier_text else " (pending Notebook 07 run)."),
     "achievable": "Every document already exists as a notebook output; no new artifact type required.",
     "relevant": "SR 11-7 / OCC 2011-12 examinations expect exactly this kind of end-to-end evidentiary trail.",
     "time_bound": "Documentation package finalized before go-live (end of Phase 6)."},
]

smart_df = pd.DataFrame(SMART_RECOMMENDATIONS)
smart_path = EXEC_DIR / "smart_recommendations_by_team.csv"
smart_df.to_csv(smart_path, index=False)

for rec in SMART_RECOMMENDATIONS:
    print(f"\n{rec['team']}")
    print(f"  Specific  : {rec['specific']}")
    print(f"  Measurable: {rec['measurable']}")
print(f"\n\u2705 Saved -> {smart_path} ({len(SMART_RECOMMENDATIONS)} teams)")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: CHARTS -- 6 STATIC CHARTS FOR THE WORD REPORT (TITLES, AXES, LEGENDS, DATA LABELS)
# =============================================================================
_section("SECTION 8: Charts -- 6 Static Charts for the Word Report")

# --- Static PNGs cannot have hover tooltips -- a printed image has no
#     interactivity -- so every chart below carries printed data-value labels
#     directly on the bars/points instead. The interactive HTML dashboard
#     (Section 10) has real hover tooltips via Chart.js. Every chart title is
#     prefixed with the platform's stated problem-statement name, per the
#     requested chart-quality standard. ---
PROBLEM_NAME = "Phase 1 \u00b7 Problem 1 -- Credit Scoring / PD Prediction"

VIZ = {
    "surface": "#fcfcfb", "text_primary": "#0b0b0b", "text_secondary": "#52514e", "grid": "#e3e2dd",
    "cat_blue": "#2a78d6", "cat_red": "#e34948", "cat_green": "#3a9e5f", "cat_amber": "#d99a2b",
    "cat_purple": "#7d5fd6",
}


def _style_axes(ax):
    ax.set_facecolor(VIZ["surface"])
    ax.figure.set_facecolor(VIZ["surface"])
    ax.grid(axis="y", color=VIZ["grid"], linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(VIZ["grid"])
    ax.tick_params(colors=VIZ["text_secondary"], labelsize=9)
    ax.title.set_color(VIZ["text_primary"])
    ax.xaxis.label.set_color(VIZ["text_secondary"])
    ax.yaxis.label.set_color(VIZ["text_secondary"])


# --- Chart 1: cumulative net benefit over 60 months, base scenario, payback marker ---
fig, ax = plt.subplots(figsize=(9, 5.5), dpi=150)
ax.plot(monthly_model_df["month"], monthly_model_df["cumulative_net_usd"], color=VIZ["cat_blue"], linewidth=2.2, zorder=4)
ax.axhline(0, color=VIZ["text_secondary"], linewidth=0.9, zorder=2)
if _base_payback:
    ax.axvline(_base_payback, color=VIZ["cat_red"], linewidth=1.4, linestyle="--", zorder=3)
    ax.annotate(f"Payback: month {_base_payback}", xy=(_base_payback, 0), xytext=(_base_payback + 2, monthly_model_df["cumulative_net_usd"].max() * 0.6),
                color=VIZ["cat_red"], fontsize=9, arrowprops=dict(arrowstyle="->", color=VIZ["cat_red"]))
_style_axes(ax)
ax.set_xlabel("Month since project start")
ax.set_ylabel("Cumulative net benefit (USD)")
ax.set_title(f"{PROBLEM_NAME}\nCumulative Net Benefit Over 5 Years (Base Scenario)", fontsize=11)
fig.tight_layout()
chart1_path = EXEC_DIR / "financial_cumulative_net_benefit_chart.png"
fig.savefig(chart1_path, dpi=150, facecolor=VIZ["surface"])
plt.show()
plt.close(fig)
print(f"\u2705 Saved -> {chart1_path}")

# --- Chart 2: ROI% by horizon, base scenario, with data labels ---
_h_base = horizon_df[horizon_df["scenario"] == "base"]
fig, ax = plt.subplots(figsize=(8, 5.5), dpi=150)
_bars = ax.bar(_h_base["horizon_label"], _h_base["roi_pct"], color=VIZ["cat_green"], zorder=3)
ax.bar_label(_bars, fmt="%.0f%%", padding=3, color=VIZ["text_primary"], fontsize=9)
_style_axes(ax)
ax.set_xlabel("Time horizon")
ax.set_ylabel("ROI (%)")
ax.set_title(f"{PROBLEM_NAME}\nReturn on Investment by Time Horizon (Base Scenario)", fontsize=11)
fig.tight_layout()
chart2_path = EXEC_DIR / "financial_roi_by_horizon_chart.png"
fig.savefig(chart2_path, dpi=150, facecolor=VIZ["surface"])
plt.show()
plt.close(fig)
print(f"\u2705 Saved -> {chart2_path}")

# --- Chart 3: investment breakdown, horizontal bar with data labels ---
_inv_sorted = investment_df.sort_values("amount_usd")
fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
_bars = ax.barh(_inv_sorted["category"], _inv_sorted["amount_usd"], color=VIZ["cat_purple"], zorder=3)
ax.bar_label(_bars, fmt="$%.0f", padding=3, color=VIZ["text_primary"], fontsize=8)
_style_axes(ax)
ax.grid(axis="x", color=VIZ["grid"], linewidth=0.8, zorder=0)
ax.grid(axis="y", visible=False)
ax.set_xlabel("One-time investment (USD)")
ax.set_title(f"{PROBLEM_NAME}\nOne-Time Investment Breakdown (Total ${TOTAL_ONE_TIME_INVESTMENT_USD:,})", fontsize=11)
fig.tight_layout()
chart3_path = EXEC_DIR / "financial_investment_breakdown_chart.png"
fig.savefig(chart3_path, dpi=150, facecolor=VIZ["surface"])
plt.show()
plt.close(fig)
print(f"\u2705 Saved -> {chart3_path}")

# --- Chart 4: annual benefit breakdown, stacked bar, years 1-5, base scenario ---
_annual_rows = []
for _yr in range(1, 6):
    _m = _yr * 12
    _row = monthly_model_df[monthly_model_df["month"] == _m].iloc[0]
    _prev_m = (_yr - 1) * 12
    _prev_benefit = monthly_model_df[monthly_model_df["month"] == _prev_m].iloc[0]["cumulative_benefit_usd"] if _prev_m > 0 else 0.0
    _yr_benefit = _row["cumulative_benefit_usd"] - _prev_benefit
    _yr_loss_avoided_share = ANNUAL_LOSS_AVOIDED_USD / ANNUAL_TOTAL_BENEFIT_USD if ANNUAL_TOTAL_BENEFIT_USD > 0 else 0
    _annual_rows.append({"year": f"Year {_yr}", "loss_avoided_usd": _yr_benefit * _yr_loss_avoided_share,
                          "revenue_uplift_usd": _yr_benefit * (1 - _yr_loss_avoided_share)})
_annual_df = pd.DataFrame(_annual_rows)
fig, ax = plt.subplots(figsize=(8, 5.5), dpi=150)
_b1 = ax.bar(_annual_df["year"], _annual_df["loss_avoided_usd"], color=VIZ["cat_blue"], zorder=3, label="Loss avoided")
_b2 = ax.bar(_annual_df["year"], _annual_df["revenue_uplift_usd"], bottom=_annual_df["loss_avoided_usd"],
             color=VIZ["cat_amber"], zorder=3, label="Revenue uplift")
for _yr_i, _row in _annual_df.iterrows():
    _total = _row["loss_avoided_usd"] + _row["revenue_uplift_usd"]
    ax.text(_yr_i, _total, f"${_total:,.0f}", ha="center", va="bottom", fontsize=8, color=VIZ["text_primary"])
_style_axes(ax)
ax.set_xlabel("Year")
ax.set_ylabel("Annual benefit (USD)")
ax.set_title(f"{PROBLEM_NAME}\nAnnual Benefit Breakdown, Years 1-5 (Base Scenario)", fontsize=11)
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
chart4_path = EXEC_DIR / "financial_annual_benefit_breakdown_chart.png"
fig.savefig(chart4_path, dpi=150, facecolor=VIZ["surface"])
plt.show()
plt.close(fig)
print(f"\u2705 Saved -> {chart4_path}")

# --- Chart 5: capture-rate comparison -- legacy (assumption) vs new model (measured) ---
fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=150)
_cats = ["Legacy Model\n(ASSUMPTION)", f"{CHAMPION_NAME}\n(MEASURED, Notebook 05)"]
_vals = [SCENARIO_ASSUMPTIONS["legacy_model_top4pct_capture_rate"], CHAMPION_TOP4PCT_CAPTURE]
_bars = ax.bar(_cats, _vals, color=[VIZ["text_secondary"], VIZ["cat_green"]], zorder=3)
ax.bar_label(_bars, fmt="%.1f%%", labels=[f"{v:.1%}" for v in _vals], padding=3, color=VIZ["text_primary"], fontsize=10)
_style_axes(ax)
ax.set_ylabel("Top-4% default capture rate")
ax.set_title(f"{PROBLEM_NAME}\nDefault-Capture Rate: Legacy Assumption vs. Measured Champion", fontsize=11)
fig.tight_layout()
chart5_path = EXEC_DIR / "financial_capture_rate_comparison_chart.png"
fig.savefig(chart5_path, dpi=150, facecolor=VIZ["surface"])
plt.show()
plt.close(fig)
print(f"\u2705 Saved -> {chart5_path}")

# --- Chart 6: implementation timeline, floating horizontal bars by phase ---
fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
_phase_labels = [p["chart_label"] for p in IMPLEMENTATION_PHASES][::-1]
_starts = [p["start_week"] for p in IMPLEMENTATION_PHASES][::-1]
_durations = [p["weeks"] for p in IMPLEMENTATION_PHASES][::-1]
ax.barh(_phase_labels, _durations, left=_starts, color=VIZ["cat_blue"], zorder=3)
for _i, (_s, _d) in enumerate(zip(_starts, _durations)):
    ax.text(_s + _d / 2, _i, f"{_d}w", ha="center", va="center", fontsize=8, color="white", zorder=4)
_style_axes(ax)
ax.grid(axis="x", color=VIZ["grid"], linewidth=0.8, zorder=0)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Project week")
ax.set_title(f"{PROBLEM_NAME}\nImplementation Timeline ({IMPLEMENTATION_WEEKS} weeks total)", fontsize=11)
fig.tight_layout()
chart6_path = EXEC_DIR / "financial_implementation_timeline_chart.png"
fig.savefig(chart6_path, dpi=150, facecolor=VIZ["surface"])
plt.show()
plt.close(fig)
print(f"\u2705 Saved -> {chart6_path}")

print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: WORD REPORT -- FINANCIAL_IMPACT_REPORT.DOCX
# =============================================================================
_section("SECTION 9: Word Report -- Financial_Impact_Report.docx")


def _add_heading(doc, text, level=1):
    return doc.add_heading(text, level=level)


def _add_kv_table(doc, data: dict):
    table = doc.add_table(rows=0, cols=2)
    table.style = "Light Grid Accent 1"
    for k, v in data.items():
        row = table.add_row().cells
        row[0].text = str(k).replace("_", " ").title()
        row[1].text = str(v)
    return table


report = Document()
report.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
report.add_paragraph("Financial Impact Report & SMART Team Recommendations -- Notebook 14 (Executive Reports)")
report.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
report.add_paragraph(
    "IMPORTANT: this report combines two different kinds of numbers. Champion model performance (holdout "
    "AUC, AMEX metric, Top-4% capture) and the live portfolio default rate are MEASURED -- computed by this "
    "platform's own code from the real Kaggle AMEX data. Every dollar figure (portfolio size, costs, "
    "revenue-per-account, staffing) rests on an explicit, stated ASSUMPTION -- the Kaggle dataset contains no "
    "real financial data for any institution. This is a worked financial-impact METHODOLOGY with editable "
    "inputs, not investment, accounting, or regulatory-capital advice, and not a substitute for your own "
    "institution's real financial figures."
)

_add_heading(report, "1. Executive Summary", level=1)
report.add_paragraph(
    f"Deploying the {CHAMPION_NAME} champion model (measured holdout AMEX metric {CHAMPION_AMEX_METRIC:.4f}, "
    f"Top-4% default capture {CHAMPION_TOP4PCT_CAPTURE:.1%}) into the stated hypothetical production scenario "
    f"below is modeled to require ${TOTAL_ONE_TIME_INVESTMENT_USD:,} in one-time investment plus "
    f"${TOTAL_MONTHLY_RECURRING_COST_USD:,}/month ongoing, reaching payback in "
    f"{(str(_base_payback) + ' months') if _base_payback else 'more than 60 months'} and a 5-year cumulative "
    f"net benefit of ${_base_5yr['cumulative_net_usd']:,.0f} (base scenario; see Section 5 for conservative/"
    f"optimistic variants)."
)

_add_heading(report, "2. Methodology & Assumptions", level=1)
report.add_paragraph("MEASURED inputs (computed live by this platform's notebooks):", style="List Bullet")
_add_kv_table(report, {
    "champion_model": CHAMPION_NAME, "holdout_auc": f"{CHAMPION_HOLDOUT_AUC:.4f}",
    "holdout_amex_metric": f"{CHAMPION_AMEX_METRIC:.4f}", "holdout_top4pct_capture": f"{CHAMPION_TOP4PCT_CAPTURE:.4%}",
    "live_portfolio_default_rate": f"{LIVE_PORTFOLIO_DEFAULT_RATE:.4%}",
    "measured_from": f"{_champion_source}; train_labels.csv (this run)",
})
report.add_paragraph("ASSUMPTION inputs (stated, editable -- Sections 4-5 of the notebook):", style="List Bullet")
_assump_table_data = {k: (f"{v:,}" if isinstance(v, int) else f"{v:.2%}" if v < 1 else f"${v:,.2f}")
                       for k, v in SCENARIO_ASSUMPTIONS.items()}
_add_kv_table(report, _assump_table_data)

_add_heading(report, "3. Investment Breakdown", level=1)
_inv_table = report.add_table(rows=1, cols=2)
_inv_table.style = "Light Grid Accent 1"
_inv_table.rows[0].cells[0].text, _inv_table.rows[0].cells[1].text = "Category", "Amount (USD)"
for r in INVESTMENT_BREAKDOWN:
    c = _inv_table.add_row().cells
    c[0].text, c[1].text = r["category"], f"${r['amount_usd']:,}"
report.add_paragraph(f"Total one-time investment: ${TOTAL_ONE_TIME_INVESTMENT_USD:,}")
report.add_paragraph(f"Total monthly recurring cost post go-live: ${TOTAL_MONTHLY_RECURRING_COST_USD:,}/month")
report.add_picture(str(chart3_path), width=Inches(6.0))

_add_heading(report, "4. Financial Projections (Base Scenario)", level=1)
_proj_table = report.add_table(rows=1, cols=5)
_proj_table.style = "Light Grid Accent 1"
_hdr = _proj_table.rows[0].cells
_hdr[0].text, _hdr[1].text, _hdr[2].text, _hdr[3].text, _hdr[4].text = \
    "Horizon", "Cumulative Cost", "Cumulative Benefit", "Cumulative Net", "ROI %"
for _, r in _h_base.iterrows():
    c = _proj_table.add_row().cells
    c[0].text = r["horizon_label"]
    c[1].text = f"${r['cumulative_cost_usd']:,.0f}"
    c[2].text = f"${r['cumulative_benefit_usd']:,.0f}"
    c[3].text = f"${r['cumulative_net_usd']:,.0f}"
    c[4].text = f"{r['roi_pct']:.0f}%"
report.add_picture(str(chart1_path), width=Inches(6.0))
report.add_picture(str(chart2_path), width=Inches(6.0))
report.add_picture(str(chart4_path), width=Inches(6.0))

_add_heading(report, "5. Scenario Comparison (Conservative / Base / Optimistic)", level=1)
report.add_paragraph(
    "Scenarios scale the BENEFIT side only (conservative x0.7, base x1.0, optimistic x1.3) -- costs are held "
    "fixed across scenarios since they represent up-front staffing/infrastructure commitments, not outcomes "
    "that vary with deployment performance."
)
_scen_table = report.add_table(rows=1, cols=4)
_scen_table.style = "Light Grid Accent 1"
_hdr = _scen_table.rows[0].cells
_hdr[0].text, _hdr[1].text, _hdr[2].text, _hdr[3].text = "Scenario", "5-Year Net Benefit", "5-Year ROI %", "Payback (months)"
for _sname in ("conservative", "base", "optimistic"):
    _r5 = horizon_df[(horizon_df["scenario"] == _sname) & (horizon_df["horizon_label"] == "5 Years")].iloc[0]
    c = _scen_table.add_row().cells
    c[0].text = _sname.title()
    c[1].text = f"${_r5['cumulative_net_usd']:,.0f}"
    c[2].text = f"{_r5['roi_pct']:.0f}%"
    c[3].text = str(PAYBACK_MONTH_BY_SCENARIO[_sname]) if PAYBACK_MONTH_BY_SCENARIO[_sname] else "not reached"

_add_heading(report, "6. Model Performance Basis (Measured)", level=1)
report.add_picture(str(chart5_path), width=Inches(5.5))
report.add_paragraph(
    f"The financial case's loss-avoidance component depends on the measured improvement in Top-4% default "
    f"capture over the assumed legacy baseline: {CAPTURE_RATE_IMPROVEMENT:+.2%} percentage points."
)

_add_heading(report, "7. Implementation Timeline", level=1)
report.add_picture(str(chart6_path), width=Inches(6.0))
_tl_table = report.add_table(rows=1, cols=3)
_tl_table.style = "Light Grid Accent 1"
_hdr = _tl_table.rows[0].cells
_hdr[0].text, _hdr[1].text, _hdr[2].text = "Phase", "Start Week", "Duration (weeks)"
for p in IMPLEMENTATION_PHASES:
    c = _tl_table.add_row().cells
    c[0].text, c[1].text, c[2].text = p["phase"], str(p["start_week"]), str(p["weeks"])
report.add_paragraph(f"Total implementation duration: {IMPLEMENTATION_WEEKS} weeks (~{IMPLEMENTATION_WEEKS / 4.345:.1f} months)")

_add_heading(report, "8. Ongoing Monitoring Cost", level=1)
report.add_paragraph(
    f"${TOTAL_MONTHLY_RECURRING_COST_USD:,}/month post go-live, covering cloud hosting/inference, MRM "
    f"oversight, and amortized quarterly retraining (breakdown in Section 5's monthly recurring cost table)."
)

_add_heading(report, "9. SMART Recommendations by Team", level=1)
for rec in SMART_RECOMMENDATIONS:
    report.add_heading(rec["team"], level=2)
    _add_kv_table(report, {k: v for k, v in rec.items() if k != "team"})

_add_heading(report, "10. Limitations & Disclaimers", level=1)
for _d in [
    "All dollar figures rest on the stated hypothetical scenario in Sections 4-5, not real AMEX financials -- "
    "the Kaggle dataset contains no revenue, cost, or investment data for any institution.",
    "This report is a worked methodology with editable assumption constants, not investment, accounting, tax, "
    "or regulatory advice. Consult your institution's finance, actuarial, and compliance functions before "
    "acting on any figure in this report.",
    "Benefit realization depends on operational execution (e.g. Collections actually acting on model flags) "
    "that this report cannot guarantee -- see the SMART recommendations above for the operational prerequisites.",
]:
    report.add_paragraph(_d, style="List Bullet")

report_path = EXEC_DIR / "Financial_Impact_Report.docx"
report.save(str(report_path))
print(f"\u2705 Saved -> {report_path}")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: INTERACTIVE HTML DASHBOARD -- FINANCIAL_IMPACT_DASHBOARD.HTML
# =============================================================================
_section("SECTION 10: Interactive HTML Dashboard -- Financial_Impact_Dashboard.html")

# --- Dynamic data is embedded as one JSON blob and substituted into the
#     template via a plain string token -- NOT an f-string -- so none of the
#     CSS/JS below needs any brace-escaping. All interactivity (horizon
#     slicer, scenario selector, team search/filter, sortable table) runs
#     client-side in vanilla JS against this embedded data; no server, no
#     network calls except the one-time Chart.js CDN load. ---
dashboard_payload = {
    "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "problem_name": PROBLEM_NAME,
    "champion_model": CHAMPION_NAME,
    "measured": {
        "holdout_auc": round(CHAMPION_HOLDOUT_AUC, 4), "holdout_amex_metric": round(CHAMPION_AMEX_METRIC, 4),
        "holdout_top4pct_capture": round(CHAMPION_TOP4PCT_CAPTURE, 4),
        "live_portfolio_default_rate": round(LIVE_PORTFOLIO_DEFAULT_RATE, 4),
    },
    "assumptions": SCENARIO_ASSUMPTIONS,
    "legacy_capture_rate": SCENARIO_ASSUMPTIONS["legacy_model_top4pct_capture_rate"],
    "investment_breakdown": INVESTMENT_BREAKDOWN,
    "total_one_time_investment": TOTAL_ONE_TIME_INVESTMENT_USD,
    "monthly_recurring_cost": TOTAL_MONTHLY_RECURRING_COST_USD,
    "implementation_weeks": IMPLEMENTATION_WEEKS,
    "implementation_phases": IMPLEMENTATION_PHASES,
    "monthly_models": {name: df[["month", "month_cost_usd", "month_benefit_usd", "cumulative_cost_usd",
                                  "cumulative_benefit_usd", "cumulative_net_usd", "roi_pct"]].to_dict("records")
                        for name, df in monthly_model_by_scenario.items()},
    "horizons": [{"label": lbl, "months": m} for lbl, m in HORIZONS],
    "payback_by_scenario": PAYBACK_MONTH_BY_SCENARIO,
    "smart_recommendations": SMART_RECOMMENDATIONS,
    "risk_tier": NB07_SUMMARY.get("risk_tier") if NB07_SUMMARY else None,
    "psi_significant_features": NB07_SUMMARY.get("psi_significant_shift_features") if NB07_SUMMARY else None,
    "nb07_available": NB07_SUMMARY is not None,
}
dashboard_json = json.dumps(dashboard_payload)

HTML_TEMPLATE = """<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>AMEX Credit Risk Platform - Financial Impact Dashboard</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.0/chart.umd.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/chartjs-plugin-datalabels/2.2.0/chartjs-plugin-datalabels.min.js"></script>
<style>
  :root {
    --bg: #f5f5f3; --surface: #ffffff; --ink: #14140f; --ink-soft: #55544c; --grid: #e2e0d8;
    --blue: #2a78d6; --red: #e34948; --green: #3a9e5f; --amber: #d99a2b; --purple: #7d5fd6;
  }
  * { box-sizing: border-box; }
  body { margin: 0; font-family: -apple-system, Segoe UI, Roboto, Helvetica, Arial, sans-serif;
         background: var(--bg); color: var(--ink); }
  header { padding: 24px 28px 18px; background: var(--surface); border-bottom: 1px solid var(--grid); }
  header h1 { margin: 0 0 4px; font-size: 22px; }
  header p { margin: 2px 0; color: var(--ink-soft); font-size: 13px; }
  .disclaimer { background: #fff8e6; border: 1px solid #eddca3; border-radius: 8px; padding: 12px 16px;
                margin: 16px 28px 0; font-size: 12.5px; color: #6b5610; }
  main { padding: 20px 28px 60px; max-width: 1400px; margin: 0 auto; }
  .controls { display: flex; flex-wrap: wrap; gap: 18px; align-items: center; background: var(--surface);
              border: 1px solid var(--grid); border-radius: 10px; padding: 14px 18px; margin-bottom: 20px; }
  .control-group { display: flex; flex-direction: column; gap: 6px; }
  .control-group label { font-size: 11px; text-transform: uppercase; letter-spacing: .04em; color: var(--ink-soft); }
  .btn-row { display: flex; gap: 6px; flex-wrap: wrap; }
  .btn-row button { border: 1px solid var(--grid); background: #fff; color: var(--ink); padding: 6px 12px;
                     border-radius: 20px; font-size: 12.5px; cursor: pointer; }
  .btn-row button.active { background: var(--blue); color: #fff; border-color: var(--blue); }
  select, input[type=text] { border: 1px solid var(--grid); border-radius: 6px; padding: 6px 10px; font-size: 13px; }
  .kpi-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(190px, 1fr)); gap: 14px; margin-bottom: 24px; }
  .kpi-card { background: var(--surface); border: 1px solid var(--grid); border-radius: 10px; padding: 14px 16px; }
  .kpi-card .label { font-size: 11px; text-transform: uppercase; color: var(--ink-soft); letter-spacing: .04em; }
  .kpi-card .value { font-size: 22px; font-weight: 700; margin-top: 4px; }
  .kpi-card .sub { font-size: 11px; color: var(--ink-soft); margin-top: 2px; }
  .chart-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(430px, 1fr)); gap: 18px; margin-bottom: 24px; }
  .chart-card { background: var(--surface); border: 1px solid var(--grid); border-radius: 10px; padding: 14px 16px; }
  .chart-card h3 { margin: 0 0 10px; font-size: 13.5px; }
  .chart-card canvas { max-height: 320px; }
  table { width: 100%; border-collapse: collapse; font-size: 12.5px; }
  th, td { text-align: left; padding: 8px 10px; border-bottom: 1px solid var(--grid); }
  th { cursor: pointer; user-select: none; color: var(--ink-soft); font-weight: 600; }
  th.sortable:hover { color: var(--blue); }
  .table-card { background: var(--surface); border: 1px solid var(--grid); border-radius: 10px; padding: 16px 18px; margin-bottom: 20px; }
  .table-card h3 { margin: 0 0 12px; font-size: 14px; }
  .table-scroll { overflow-x: auto; }
  .assumption-tag { display: inline-block; font-size: 10px; padding: 1px 6px; border-radius: 8px; margin-left: 6px; }
  .tag-measured { background: #e5f4ea; color: #1e7a3f; }
  .tag-assumption { background: #fdeee6; color: #a14a1a; }
  footer { padding: 20px 28px 40px; color: var(--ink-soft); font-size: 11.5px; max-width: 1400px; margin: 0 auto; }
</style>
</head>
<body>
<header>
  <h1>AMEX Enterprise Credit Risk Platform &mdash; Financial Impact Dashboard</h1>
  <p id="problemNameLine"></p>
  <p id="generatedLine"></p>
</header>
<div class="disclaimer">
  <strong>How to read this dashboard:</strong> figures tagged <span class="assumption-tag tag-measured">MEASURED</span>
  come from this platform's own live model runs. Figures tagged <span class="assumption-tag tag-assumption">ASSUMPTION</span>
  rest on a stated hypothetical deployment scenario (the Kaggle AMEX dataset has no real financial data) &mdash;
  edit the constants in Notebook 14, Sections 4-5, and re-run to substitute your institution's real figures.
  This is a worked financial methodology, not investment or accounting advice.
</div>
<main>
  <div class="controls">
    <div class="control-group">
      <label>Time Horizon</label>
      <div class="btn-row" id="horizonButtons"></div>
    </div>
    <div class="control-group">
      <label>Scenario</label>
      <select id="scenarioSelect">
        <option value="conservative">Conservative (x0.7 benefit)</option>
        <option value="base" selected>Base (x1.0 benefit)</option>
        <option value="optimistic">Optimistic (x1.3 benefit)</option>
      </select>
    </div>
    <div class="control-group">
      <label>Filter Team Recommendations</label>
      <select id="teamFilter"></select>
    </div>
    <div class="control-group">
      <label>Search Recommendations</label>
      <input type="text" id="teamSearch" placeholder="e.g. compliance, ROI, payback">
    </div>
  </div>

  <div class="kpi-grid" id="kpiGrid"></div>

  <div class="chart-grid">
    <div class="chart-card"><h3 id="chart1Title"></h3><canvas id="chartNetBenefit"></canvas></div>
    <div class="chart-card"><h3 id="chart2Title"></h3><canvas id="chartRoiByHorizon"></canvas></div>
    <div class="chart-card"><h3 id="chart3Title"></h3><canvas id="chartInvestment"></canvas></div>
    <div class="chart-card"><h3 id="chart4Title"></h3><canvas id="chartAnnualBenefit"></canvas></div>
    <div class="chart-card"><h3 id="chart5Title"></h3><canvas id="chartCaptureRate"></canvas></div>
    <div class="chart-card"><h3 id="chart6Title"></h3><canvas id="chartTimeline"></canvas></div>
  </div>

  <div class="table-card">
    <h3>Year-by-Year Financial Breakdown (click a column header to sort, selected scenario)</h3>
    <div class="table-scroll"><table id="financialTable"></table></div>
  </div>

  <div class="table-card">
    <h3>SMART Recommendations by Team</h3>
    <div class="table-scroll"><table id="smartTable"></table></div>
  </div>
</main>
<footer id="footerAssumptions"></footer>

<script>
const DATA = __DASHBOARD_DATA_JSON__;
const COLORS = { blue: "#2a78d6", red: "#e34948", green: "#3a9e5f", amber: "#d99a2b", purple: "#7d5fd6", grey: "#8b8a82" };
if (typeof Chart !== "undefined") {
  if (typeof ChartDataLabels !== "undefined") { Chart.register(ChartDataLabels); }
  Chart.defaults.font.size = 11;
  Chart.defaults.color = "#55544c";
}

document.getElementById("problemNameLine").textContent = DATA.problem_name;
document.getElementById("generatedLine").textContent = "Generated " + DATA.generated_at +
  " | Champion model: " + DATA.champion_model + " (measured, Notebook 05)" +
  (DATA.nb07_available ? " | Model risk tier: " + DATA.risk_tier + " (Notebook 07)" : " | Notebook 07 not yet run");

let state = { scenario: "base", horizonMonths: 60 };

function fmtUsd(v) {
  if (v === null || v === undefined) return "n/a";
  const sign = v < 0 ? "-" : "";
  return sign + "$" + Math.abs(Math.round(v)).toLocaleString();
}
function fmtPct(v) { return (v === null || v === undefined) ? "n/a" : v.toFixed(0) + "%"; }

function monthlyRows() { return DATA.monthly_models[state.scenario]; }
function rowAtMonth(m) {
  const rows = monthlyRows();
  return rows.find(function(r) { return r.month === m; }) || rows[rows.length - 1];
}

function renderHorizonButtons() {
  const el = document.getElementById("horizonButtons");
  el.innerHTML = "";
  DATA.horizons.forEach(function(h) {
    const b = document.createElement("button");
    b.textContent = h.label;
    b.className = (h.months === state.horizonMonths) ? "active" : "";
    b.onclick = function() { state.horizonMonths = h.months; renderAll(); };
    el.appendChild(b);
  });
}

function renderKpis() {
  const r = rowAtMonth(state.horizonMonths);
  const payback = DATA.payback_by_scenario[state.scenario];
  const captureImprovement = Math.max(0, DATA.measured.holdout_top4pct_capture - DATA.legacy_capture_rate);
  const cards = [
    { label: "Total One-Time Investment", value: fmtUsd(DATA.total_one_time_investment), sub: "ASSUMPTION" },
    { label: "Monthly Recurring Cost", value: fmtUsd(DATA.monthly_recurring_cost) + "/mo", sub: "ASSUMPTION" },
    { label: "Cumulative Net Benefit", value: fmtUsd(r.cumulative_net_usd), sub: "at selected horizon, " + state.scenario },
    { label: "ROI at Horizon", value: fmtPct(r.roi_pct), sub: "at selected horizon, " + state.scenario },
    { label: "Payback Period", value: payback ? (payback + " months") : "not reached (60mo)", sub: state.scenario + " scenario" },
    { label: "Implementation Duration", value: DATA.implementation_weeks + " weeks", sub: "stated project plan" },
    { label: "Capture-Rate Improvement", value: (captureImprovement * 100).toFixed(1) + " pts", sub: "MEASURED vs legacy ASSUMPTION" },
  ];
  const grid = document.getElementById("kpiGrid");
  grid.innerHTML = "";
  cards.forEach(function(c) {
    const div = document.createElement("div");
    div.className = "kpi-card";
    div.innerHTML = "<div class=\\"label\\">" + c.label + "</div><div class=\\"value\\">" + c.value + "</div><div class=\\"sub\\">" + c.sub + "</div>";
    grid.appendChild(div);
  });
}

let charts = {};
function destroyChart(key) { if (charts[key]) { charts[key].destroy(); } }

function renderNetBenefitChart() {
  destroyChart("net");
  const rows = monthlyRows().filter(function(r) { return r.month <= state.horizonMonths; });
  document.getElementById("chart1Title").textContent = DATA.problem_name + ": Cumulative Net Benefit Over Time";
  const ctx = document.getElementById("chartNetBenefit").getContext("2d");
  charts.net = new Chart(ctx, { type: "line",
    data: { labels: rows.map(function(r) { return "M" + r.month; }),
            datasets: [{ label: "Cumulative net benefit (USD)", data: rows.map(function(r) { return r.cumulative_net_usd; }),
                         borderColor: COLORS.blue, backgroundColor: COLORS.blue, tension: 0.15, pointRadius: 0 }] },
    options: { plugins: { legend: { display: true }, datalabels: { display: false },
                           tooltip: { callbacks: { label: function(c) { return fmtUsd(c.parsed.y); } } } },
               scales: { x: { title: { display: true, text: "Month since project start" } },
                         y: { title: { display: true, text: "Cumulative net benefit (USD)" } } } } });
}

function renderRoiByHorizonChart() {
  destroyChart("roi");
  document.getElementById("chart2Title").textContent = DATA.problem_name + ": ROI by Time Horizon";
  const rows = DATA.horizons.map(function(h) {
    return { label: h.label, roi: rowAtMonth(h.months).roi_pct };
  });
  const ctx = document.getElementById("chartRoiByHorizon").getContext("2d");
  charts.roi = new Chart(ctx, { type: "bar",
    data: { labels: rows.map(function(r) { return r.label; }),
            datasets: [{ label: "ROI %", data: rows.map(function(r) { return r.roi; }), backgroundColor: COLORS.green }] },
    options: { plugins: { legend: { display: true },
                           datalabels: { anchor: "end", align: "top", formatter: function(v) { return v.toFixed(0) + "%"; } },
                           tooltip: { callbacks: { label: function(c) { return fmtPct(c.parsed.y); } } } },
               scales: { x: { title: { display: true, text: "Time horizon" } },
                         y: { title: { display: true, text: "ROI (%)" } } } } });
}

function renderInvestmentChart() {
  destroyChart("inv");
  document.getElementById("chart3Title").textContent = DATA.problem_name + ": One-Time Investment Breakdown (ASSUMPTION)";
  const ctx = document.getElementById("chartInvestment").getContext("2d");
  const palette = [COLORS.blue, COLORS.red, COLORS.green, COLORS.amber, COLORS.purple, COLORS.grey];
  charts.inv = new Chart(ctx, { type: "doughnut",
    data: { labels: DATA.investment_breakdown.map(function(r) { return r.category; }),
            datasets: [{ data: DATA.investment_breakdown.map(function(r) { return r.amount_usd; }), backgroundColor: palette }] },
    options: { plugins: { legend: { display: true, position: "bottom", labels: { boxWidth: 10, font: { size: 9 } } },
                           datalabels: { formatter: function(v) { return fmtUsd(v); }, font: { size: 9 } },
                           tooltip: { callbacks: { label: function(c) { return c.label + ": " + fmtUsd(c.parsed); } } } } } });
}

function renderAnnualBenefitChart() {
  destroyChart("annual");
  document.getElementById("chart4Title").textContent = DATA.problem_name + ": Annual Benefit Breakdown, Years 1-5";
  const rows = monthlyRows();
  const years = [1, 2, 3, 4, 5];
  const totalBenefit = DATA.monthly_models.base[DATA.monthly_models.base.length - 1].cumulative_benefit_usd;
  const lossShare = totalBenefit > 0 ? (rowAtMonth(12).cumulative_benefit_usd > 0 ? 0.5 : 0.5) : 0.5;
  const perYear = years.map(function(y) {
    const cur = rows.find(function(r) { return r.month === y * 12; }) || rows[rows.length - 1];
    const prevMonth = (y - 1) * 12;
    const prev = prevMonth > 0 ? (rows.find(function(r) { return r.month === prevMonth; }) || { cumulative_benefit_usd: 0 }) : { cumulative_benefit_usd: 0 };
    return cur.cumulative_benefit_usd - prev.cumulative_benefit_usd;
  });
  const ctx = document.getElementById("chartAnnualBenefit").getContext("2d");
  charts.annual = new Chart(ctx, { type: "bar",
    data: { labels: years.map(function(y) { return "Year " + y; }),
            datasets: [{ label: "Total annual benefit (USD)", data: perYear, backgroundColor: COLORS.amber }] },
    options: { plugins: { legend: { display: true },
                           datalabels: { anchor: "end", align: "top", formatter: function(v) { return fmtUsd(v); }, font: { size: 9 } },
                           tooltip: { callbacks: { label: function(c) { return fmtUsd(c.parsed.y); } } } },
               scales: { x: { title: { display: true, text: "Year" } },
                         y: { title: { display: true, text: "Annual benefit (USD)" } } } } });
}

function renderCaptureRateChart() {
  destroyChart("capture");
  document.getElementById("chart5Title").textContent = DATA.problem_name + ": Default-Capture Rate, Legacy vs Measured Champion";
  const ctx = document.getElementById("chartCaptureRate").getContext("2d");
  charts.capture = new Chart(ctx, { type: "bar",
    data: { labels: ["Legacy Model (ASSUMPTION)", DATA.champion_model + " (MEASURED)"],
            datasets: [{ label: "Top-4% default capture rate", data: [DATA.legacy_capture_rate * 100, DATA.measured.holdout_top4pct_capture * 100],
                         backgroundColor: [COLORS.grey, COLORS.green] }] },
    options: { plugins: { legend: { display: false },
                           datalabels: { anchor: "end", align: "top", formatter: function(v) { return v.toFixed(1) + "%"; } },
                           tooltip: { callbacks: { label: function(c) { return c.parsed.y.toFixed(1) + "%"; } } } },
               scales: { y: { title: { display: true, text: "Capture rate (%)" } } } } });
}

function renderTimelineChart() {
  destroyChart("timeline");
  document.getElementById("chart6Title").textContent = DATA.problem_name + ": Implementation Timeline (" + DATA.implementation_weeks + " weeks)";
  const ctx = document.getElementById("chartTimeline").getContext("2d");
  const phases = DATA.implementation_phases;
  charts.timeline = new Chart(ctx, { type: "bar",
    data: { labels: phases.map(function(p) { return p.chart_label; }),
            datasets: [{ label: "Weeks", data: phases.map(function(p) { return [p.start_week, p.start_week + p.weeks]; }),
                         backgroundColor: COLORS.blue }] },
    options: { indexAxis: "y",
               plugins: { legend: { display: false },
                          datalabels: { formatter: function(v) { return (v[1] - v[0]) + "w"; }, color: "#fff", font: { size: 9 } },
                          tooltip: { callbacks: {
                            title: function(items) { return phases[items[0].dataIndex].phase; },
                            label: function(c) { return "Weeks " + c.raw[0] + "-" + c.raw[1]; } } } },
               scales: { x: { title: { display: true, text: "Project week" } } } } });
}

function renderFinancialTable() {
  const rows = DATA.horizons.map(function(h) { return rowAtMonth(h.months); });
  const labels = DATA.horizons.map(function(h) { return h.label; });
  const table = document.getElementById("financialTable");
  let sortState = { col: null, dir: 1 };
  function draw() {
    let data = rows.map(function(r, i) { return { label: labels[i], r: r }; });
    if (sortState.col) {
      data.sort(function(a, b) {
        const va = sortState.col === "label" ? a.label : a.r[sortState.col];
        const vb = sortState.col === "label" ? b.label : b.r[sortState.col];
        return va > vb ? sortState.dir : va < vb ? -sortState.dir : 0;
      });
    }
    let html = "<thead><tr>" +
      "<th class=\\"sortable\\" data-col=\\"label\\">Horizon</th>" +
      "<th class=\\"sortable\\" data-col=\\"cumulative_cost_usd\\">Cumulative Cost</th>" +
      "<th class=\\"sortable\\" data-col=\\"cumulative_benefit_usd\\">Cumulative Benefit</th>" +
      "<th class=\\"sortable\\" data-col=\\"cumulative_net_usd\\">Cumulative Net</th>" +
      "<th class=\\"sortable\\" data-col=\\"roi_pct\\">ROI %</th></tr></thead><tbody>";
    data.forEach(function(d) {
      html += "<tr><td>" + d.label + "</td><td>" + fmtUsd(d.r.cumulative_cost_usd) + "</td><td>" +
        fmtUsd(d.r.cumulative_benefit_usd) + "</td><td>" + fmtUsd(d.r.cumulative_net_usd) + "</td><td>" +
        fmtPct(d.r.roi_pct) + "</td></tr>";
    });
    html += "</tbody>";
    table.innerHTML = html;
    table.querySelectorAll("th.sortable").forEach(function(th) {
      th.onclick = function() {
        const col = th.getAttribute("data-col");
        sortState.dir = (sortState.col === col) ? -sortState.dir : 1;
        sortState.col = col;
        draw();
      };
    });
  }
  draw();
}

function renderSmartTable() {
  const teamSel = document.getElementById("teamFilter");
  const search = document.getElementById("teamSearch").value.toLowerCase();
  const teamValue = teamSel.value;
  const table = document.getElementById("smartTable");
  let rows = DATA.smart_recommendations;
  if (teamValue !== "__all__") { rows = rows.filter(function(r) { return r.team === teamValue; }); }
  if (search) {
    rows = rows.filter(function(r) {
      return (r.team + " " + r.specific + " " + r.measurable + " " + r.achievable + " " + r.relevant + " " + r.time_bound).toLowerCase().indexOf(search) !== -1;
    });
  }
  let html = "<thead><tr><th>Team</th><th>Specific</th><th>Measurable</th><th>Achievable</th><th>Relevant</th><th>Time-Bound</th></tr></thead><tbody>";
  rows.forEach(function(r) {
    html += "<tr><td><strong>" + r.team + "</strong></td><td>" + r.specific + "</td><td>" + r.measurable +
      "</td><td>" + r.achievable + "</td><td>" + r.relevant + "</td><td>" + r.time_bound + "</td></tr>";
  });
  html += "</tbody>";
  table.innerHTML = html;
}

function populateTeamFilter() {
  const sel = document.getElementById("teamFilter");
  sel.innerHTML = "<option value=\\"__all__\\">All Teams</option>";
  DATA.smart_recommendations.forEach(function(r) {
    const opt = document.createElement("option");
    opt.value = r.team; opt.textContent = r.team;
    sel.appendChild(opt);
  });
}

function renderFooter() {
  const el = document.getElementById("footerAssumptions");
  let html = "<strong>Stated assumption constants (edit in Notebook 14, Sections 4-5, and re-run to update this dashboard):</strong><br>";
  Object.keys(DATA.assumptions).forEach(function(k) {
    html += k.replace(/_/g, " ") + " = " + DATA.assumptions[k] + " &nbsp;|&nbsp; ";
  });
  el.innerHTML = html;
}

const CHART_LIB_AVAILABLE = (typeof Chart !== "undefined");
let chartLibWarningShown = false;

function safeRenderChart(fn, label) {
  if (!CHART_LIB_AVAILABLE) {
    if (!chartLibWarningShown) {
      chartLibWarningShown = true;
      const warn = document.createElement("div");
      warn.className = "disclaimer";
      warn.style.margin = "0 0 16px";
      warn.innerHTML = "<strong>Charts unavailable:</strong> the Chart.js library did not load from its CDN " +
        "(no internet connection reached it). KPIs, tables, and filters below still work normally -- only " +
        "the 6 chart panels are affected. Reconnect to the internet and reload this page to see them.";
      document.querySelector("main").insertBefore(warn, document.querySelector(".kpi-grid"));
    }
    return;
  }
  try { fn(); } catch (e) { console.error("Chart render failed (" + label + "):", e); }
}

function renderAll() {
  renderHorizonButtons();
  renderKpis();
  safeRenderChart(renderNetBenefitChart, "net benefit");
  safeRenderChart(renderRoiByHorizonChart, "roi by horizon");
  safeRenderChart(renderInvestmentChart, "investment breakdown");
  safeRenderChart(renderAnnualBenefitChart, "annual benefit");
  safeRenderChart(renderCaptureRateChart, "capture rate");
  safeRenderChart(renderTimelineChart, "timeline");
  renderFinancialTable();
  renderSmartTable();
}

document.getElementById("scenarioSelect").addEventListener("change", function(e) { state.scenario = e.target.value; renderAll(); });
document.getElementById("teamFilter").addEventListener("change", renderSmartTable);
document.getElementById("teamSearch").addEventListener("input", renderSmartTable);

populateTeamFilter();
renderFooter();
renderAll();
</script>
</body>
</html>
"""

html_content = HTML_TEMPLATE.replace("__DASHBOARD_DATA_JSON__", dashboard_json)
dashboard_path = EXEC_DIR / "Financial_Impact_Dashboard.html"
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"Dashboard payload size: {len(dashboard_json):,} bytes of embedded JSON data")
print(f"\u2705 Saved -> {dashboard_path}")
print(f"    Open this file directly in any browser. It loads Chart.js from a CDN on first open, so an "
      f"internet connection is needed then (no network calls happen after the page loads).")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: VERIFICATION -- INTEGRITY CHECKS ON EVERYTHING THIS NOTEBOOK WROTE
# =============================================================================
_section("SECTION 11: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("champion metrics loaded (holdout AUC in [0,1])", 0.0 <= CHAMPION_HOLDOUT_AUC <= 1.0)
_check("live portfolio default rate is a plausible fraction", 0.0 < LIVE_PORTFOLIO_DEFAULT_RATE < 1.0)
_check("capture-rate improvement is non-negative (clamped)", CAPTURE_RATE_IMPROVEMENT >= 0.0)
_check("monthly financial model has 60 rows per scenario",
       all(len(df) == TOTAL_MONTHS_SIMULATED for df in monthly_model_by_scenario.values()))
_check("cumulative cost is monotonically non-decreasing (base scenario)",
       monthly_model_df["cumulative_cost_usd"].is_monotonic_increasing)
_check("cumulative benefit is monotonically non-decreasing (base scenario)",
       monthly_model_df["cumulative_benefit_usd"].is_monotonic_increasing)
_check("horizon summary covers 3 scenarios x 7 horizons", len(horizon_df) == 3 * len(HORIZONS),
       f"({len(horizon_df)})")
_check("optimistic 5yr net benefit >= base >= conservative (benefit-side scaling is monotonic)",
       horizon_df[(horizon_df.scenario == "optimistic") & (horizon_df.horizon_label == "5 Years")]["cumulative_net_usd"].iloc[0] >=
       horizon_df[(horizon_df.scenario == "base") & (horizon_df.horizon_label == "5 Years")]["cumulative_net_usd"].iloc[0] >=
       horizon_df[(horizon_df.scenario == "conservative") & (horizon_df.horizon_label == "5 Years")]["cumulative_net_usd"].iloc[0])
_check("SMART recommendations cover 10 teams", len(SMART_RECOMMENDATIONS) == 10, f"({len(SMART_RECOMMENDATIONS)})")
_check("implementation timeline weeks sum matches IMPLEMENTATION_WEEKS",
       sum(p["weeks"] for p in IMPLEMENTATION_PHASES) == IMPLEMENTATION_WEEKS)
_check("dashboard HTML contains the embedded data blob (not the raw placeholder token)",
       "__DASHBOARD_DATA_JSON__" not in html_content and CHAMPION_NAME in html_content)

_expected_files = [investment_path, timeline_path, monthly_model_path, horizon_path, smart_path,
                    chart1_path, chart2_path, chart3_path, chart4_path, chart5_path, chart6_path,
                    report_path, dashboard_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 14 verification checks failed. See \u274c lines above.")

print("\nAll Notebook 14 checks passed.")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: RESOURCE / PERFORMANCE REPORT
# =============================================================================
_section("SECTION 12: Resource / Performance Report")

_final_rss_gb = _rss_gb()
performance_report = {
    "warp_thread_count_configured": WARP_THREAD_COUNT,
    "note": "This notebook is I/O- and report-generation-bound (small CSV/JSON reads, chart rendering, "
            "docx/html writing) -- there is no large dataframe or model-inference workload here for the "
            "adaptive RAM ceiling or a GPU probe to meaningfully apply to, unlike Notebooks 05-13.",
    "live_available_ram_gb_at_start": round(LIVE_AVAILABLE_RAM_BYTES / 1e9, 2),
    "process_rss_at_start_gb": round(_process_start_rss_gb, 2),
    "process_rss_at_end_gb": round(_final_rss_gb, 2),
}
performance_report_path = ARTIFACTS_DIR / "notebook_14_performance_report.json"
with open(performance_report_path, "w", encoding="utf-8") as f:
    json.dump(performance_report, f, indent=2)

print(f"Process RSS: {_process_start_rss_gb:.2f} GB (start) -> {_final_rss_gb:.2f} GB (end)")
print(f"\u2705 Saved -> {performance_report_path}")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: WRITE NOTEBOOK 14 SUMMARY ARTIFACT (for Notebook 17's rollup)
# =============================================================================
_section("SECTION 13: Write Notebook 14 Summary Artifact")

notebook_14_summary = {
    "notebook": "14_executive_reports",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "champion_model": CHAMPION_NAME,
    "measured_holdout_amex_metric": CHAMPION_AMEX_METRIC,
    "measured_holdout_top4pct_capture": CHAMPION_TOP4PCT_CAPTURE,
    "live_portfolio_default_rate": LIVE_PORTFOLIO_DEFAULT_RATE,
    "total_one_time_investment_usd": TOTAL_ONE_TIME_INVESTMENT_USD,
    "monthly_recurring_cost_usd": TOTAL_MONTHLY_RECURRING_COST_USD,
    "implementation_weeks": IMPLEMENTATION_WEEKS,
    "base_5yr_cumulative_net_benefit_usd": float(_base_5yr["cumulative_net_usd"]),
    "base_5yr_roi_pct": float(_base_5yr["roi_pct"]),
    "payback_month_by_scenario": PAYBACK_MONTH_BY_SCENARIO,
    "smart_recommendation_teams": [r["team"] for r in SMART_RECOMMENDATIONS],
    "output_files": {p.name: str(p) for p in _expected_files + [performance_report_path]},
}
nb14_summary_path = ARTIFACTS_DIR / "notebook_14_summary.json"
with open(nb14_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_14_summary, f, indent=2)
print(f"\u2705 Saved -> {nb14_summary_path} (Notebook 17 reads this file to build the rolled-up Executive "
      f"Reports section)")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 14: Notebook 14 Complete -- Handoff to Notebook 15")

print("NOTEBOOK 14: EXECUTIVE REPORTS -- COMPLETE")
print(f"  Champion (measured)              : {CHAMPION_NAME}  (AMEX metric {CHAMPION_AMEX_METRIC:.4f})")
print(f"  Total one-time investment (ASSUMPTION): ${TOTAL_ONE_TIME_INVESTMENT_USD:,}")
print(f"  Monthly recurring cost (ASSUMPTION)   : ${TOTAL_MONTHLY_RECURRING_COST_USD:,}/mo")
print(f"  Base-scenario 5yr net benefit     : ${_base_5yr['cumulative_net_usd']:,.0f}  (ROI {_base_5yr['roi_pct']:.0f}%)")
print(f"  Base-scenario payback             : {(str(_base_payback) + ' months') if _base_payback else 'not reached within 60 months'}")
print(f"  SMART recommendations             : {len(SMART_RECOMMENDATIONS)} teams")
print(f"  Files produced                    : {len(_expected_files) + 2}")
for _p in _expected_files + [performance_report_path, nb14_summary_path]:
    print(f"    - {_p.name}")
print(f"  Next notebook                     : 15_technical_documentation.ipynb")
print("\n\u2705 Ready to proceed.")
